# Peak Detection and Behavioral Feature Extraction

Two parts, run in sequence since the second consumes the first's output:

1. **Peak identification and analysis** - finds actual high-consumption events in each customer's
   time series.
2. **Behavioral features** - converts raw consumption plus those detected peaks into one row of
   behavioral features per customer, describing consumption level, variability, time-of-day/
   weekday behavior, and peak-derived features. This is the feature matrix K-Means will train on.

Both are generic across electricity and gas via `value_col` (and, for features, `value_type` -
`'power'` for electricity's 15-min kW readings, `'energy'` for gas's already-integrated hourly kWh
readings; treating both the same way silently produces a wrong `annual_consumption` for whichever
one is actually a rate). Both use `timestamp_local` for all time-of-day logic (correct local
behavior, DST-safe).

```text
Raw consumption
      ↓
Part 1: Peak detection
      ↓
Raw consumption + peak events
      ↓
Part 2: Basic consumption features (level, variability)
      +      Time-of-day / weekday features
      +      Peak-derived features
      ↓
Customer behavioral feature matrix
```


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")
import peaks as pk
import features as feat
import time_cleaning as tc
import pandas as pd
import matplotlib.pyplot as plt

OUTPUTS_DIR = Path("../data/outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR = Path("../data/inputs/generated")

# Use time_cleaning.read_long_csv, not a raw pd.read_csv: Europe/Berlin
# timestamps have different UTC offsets across the year (DST), which a
# plain parse_dates=[...] cannot reliably reconstruct - see its docstring.
# Loaded once here, shared by both parts below.
electricity_long = tc.read_long_csv(GENERATED_DIR / "synthetic_electricity_long.csv")
gas_long = tc.read_long_csv(GENERATED_DIR / "synthetic_gas_long.csv")

print("Electricity customers:", electricity_long["customer_id"].nunique())
print("Gas customers:", gas_long["customer_id"].nunique())


# Part 1: Peak Identification and Analysis

Identifies **actual high-consumption events** in each customer's time series - deliberately
without ML. For each customer we ask: when do significant peaks occur, how high, how long, how
much above normal, and are they concentrated on weekdays/mornings/evenings?

## Method

For each customer: `baseline = median`, `threshold = 95th percentile`. Any run of consecutive
readings at or above the threshold is one peak event. Simple and auditable - refinements (rolling
baselines, hour-specific thresholds, anomaly detection) can come later once this baseline is
understood.


## Run peak detection

In [ ]:
electricity_peaks = pk.detect_all_peaks(electricity_long, value_col="power_kw")
gas_peaks = pk.detect_all_peaks(gas_long, value_col="energy_kwh")

print("Electricity peak events:", len(electricity_peaks),
      "across", electricity_peaks['customer_id'].nunique(), "customers")
print("Gas peak events:", len(gas_peaks),
      "across", gas_peaks['customer_id'].nunique(), "customers")
display(electricity_peaks.head())


## Visual sanity check

A peak detector is much easier to trust when you can see what it's doing. Shows raw consumption, baseline, threshold, and detected peak windows for two example customers.

In [ ]:
def plot_customer_peaks(customer_id, long_df, peak_df, value_col, timestamp_col="timestamp_local"):
    group = long_df[long_df["customer_id"] == customer_id].sort_values(timestamp_col)
    if group.empty:
        print("Customer not found.")
        return

    values = group[value_col].to_numpy(dtype=float)
    baseline = pk.np.nanquantile(values, pk.DEFAULT_BASELINE_QUANTILE)
    threshold = pk.np.nanquantile(values, pk.DEFAULT_PEAK_QUANTILE)

    plt.figure(figsize=(14, 5))
    plt.plot(group[timestamp_col], group[value_col], linewidth=0.7)
    plt.axhline(baseline, linestyle="--", color="green", label="Baseline (median)")
    plt.axhline(threshold, linestyle=":", color="red", label="Peak threshold (p95)")

    customer_peaks = peak_df[peak_df["customer_id"] == customer_id]
    for _, peak in customer_peaks.iterrows():
        plt.axvspan(peak["peak_start"], peak["peak_end"], alpha=0.2, color="red")

    plt.title(f"Peak detection - {customer_id}")
    plt.xlabel("Time (local)")
    plt.ylabel(value_col)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


example_customers = electricity_long["customer_id"].unique()[:2]
for cid in example_customers:
    plot_customer_peaks(cid, electricity_long, electricity_peaks, value_col="power_kw")


## Peak summary preview

In [ ]:
display(pk.summarize_peaks(electricity_peaks).head())
display(pk.summarize_peaks(gas_peaks).head())


## Save outputs (part 1)

In [ ]:
electricity_peaks_path = OUTPUTS_DIR / "electricity_peaks.csv"
gas_peaks_path = OUTPUTS_DIR / "gas_peaks.csv"

electricity_peaks.to_csv(electricity_peaks_path, index=False)
gas_peaks.to_csv(gas_peaks_path, index=False)

print("Saved:")
print(" ", electricity_peaks_path)
print(" ", gas_peaks_path)


# Part 2: Behavioral Features

Uses `electricity_peaks`/`gas_peaks` directly from Part 1 above (already tz-aware in memory - no
need to reload from the CSV just saved).

## Build the feature matrices

In [ ]:
electricity_features = feat.build_feature_matrix(electricity_long, electricity_peaks, value_col="power_kw", value_type="power")
gas_features = feat.build_feature_matrix(gas_long, gas_peaks, value_col="energy_kwh", value_type="energy")

print("Electricity feature matrix:", electricity_features.shape)
print("Gas feature matrix:", gas_features.shape)
display(electricity_features.head())


## Check for missing values

Customers with zero detected peak events have NaN peak-derived features (there's no peak to
describe) rather than being dropped. `peak_count` specifically is filled with 0 (absence of
peaks is meaningfully zero) - other peak columns (ratios, timing shares) are left as NaN, since
e.g. "mean peak ratio" is genuinely undefined with no peaks. The K-Means step handles these NaNs
via imputation.

In [ ]:
for name, df in [("electricity", electricity_features), ("gas", gas_features)]:
    df["peak_count"] = df["peak_count"].fillna(0)
    n_missing = df.isna().any(axis=1).sum()
    print(f"{name}: {n_missing} / {len(df)} customers have at least one NaN feature (customers with no detected peaks)")


## Save outputs (part 2)

In [ ]:
electricity_features_path = OUTPUTS_DIR / "electricity_features.csv"
gas_features_path = OUTPUTS_DIR / "gas_features.csv"

electricity_features.to_csv(electricity_features_path, index=False)
gas_features.to_csv(gas_features_path, index=False)

print("Saved:")
print(" ", electricity_features_path)
print(" ", gas_features_path)


# Single Test Customer - Peaks & Features

Runs the exact same peak detection and feature extraction as Parts 1-2 above, but on the single test customer notebook 00 generated (not the bulk 250) - so notebooks 04 and 05 can both load a precomputed, consistent feature set for that same customer instead of each generating and computing their own independently.

## Load the single test customer

In [ ]:
SINGLE_TEST_DIR = GENERATED_DIR / "single_test"
single_customer_id = (SINGLE_TEST_DIR / "LATEST_CUSTOMER_ID.txt").read_text().strip()

single_electricity = tc.read_long_csv(SINGLE_TEST_DIR / f"{single_customer_id}_electricity.csv")

single_gas_path = SINGLE_TEST_DIR / f"{single_customer_id}_gas.csv"
single_gas = tc.read_long_csv(single_gas_path) if single_gas_path.exists() else None

print("Single test customer:", single_customer_id)
print("Has gas:", single_gas is not None)


## Run peak detection and feature extraction

In [ ]:
single_electricity_peaks = pk.detect_all_peaks(single_electricity, value_col="power_kw")
single_electricity_features = feat.build_feature_matrix(
    single_electricity, single_electricity_peaks, value_col="power_kw", value_type="power",
)
single_electricity_features["peak_count"] = single_electricity_features["peak_count"].fillna(0)

if single_gas is not None:
    single_gas_peaks = pk.detect_all_peaks(single_gas, value_col="energy_kwh")
    single_gas_features = feat.build_feature_matrix(
        single_gas, single_gas_peaks, value_col="energy_kwh", value_type="energy",
    )
    single_gas_features["peak_count"] = single_gas_features["peak_count"].fillna(0)

print("Electricity peak events:", len(single_electricity_peaks))
display(single_electricity_features)


## Save outputs

In [ ]:
single_electricity_peaks.to_csv(SINGLE_TEST_DIR / f"{single_customer_id}_electricity_peaks.csv", index=False)
single_electricity_features.to_csv(SINGLE_TEST_DIR / f"{single_customer_id}_electricity_features.csv", index=False)

if single_gas is not None:
    single_gas_peaks.to_csv(SINGLE_TEST_DIR / f"{single_customer_id}_gas_peaks.csv", index=False)
    single_gas_features.to_csv(SINGLE_TEST_DIR / f"{single_customer_id}_gas_features.csv", index=False)

print("Saved single test customer peaks/features ->", SINGLE_TEST_DIR)
